<a href="https://colab.research.google.com/github/a238-khan/UK-inflation-forecasting-XAI/blob/main/05_ensemble_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Data Loading and Environment Setup

This section initialises the analytical environment and loads the processed ML ready dataset. A dynamic path detection mechanism ensures portability across different computing environments, facilitating reproducible research and collaborative workflows.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting & Pandas display settings
plt.style.use("seaborn-v0_8")
sns.set(font_scale=1.1)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

# Construct path to processed dataset
DATA_PATH = "/content/model_ready_dataset.csv"

# Load dataset
df = pd.read_csv(DATA_PATH)

print(f"Loaded dataset from: {DATA_PATH}")
print("Shape:", df.shape)

df.head()

Loaded dataset from: /content/model_ready_dataset.csv
Shape: (327, 18)


,date,cpi_yoy,petrol_yoy,bank_rate,fx_usd,gva_growth,unemp_rate,cpi_yoy_l1,cpi_yoy_l3,cpi_yoy_l6,petrol_yoy_l1,fx_usd_l1,bank_rate_l1,gva_growth_l1,unemp_rate_l1,cpi_t1,cpi_t3,cpi_t5
0,1997-12-01,1.7,3.4,5.9375,1.6597,1.4,6.4,1.9,1.8,1.7,7.5,1.6890,5.9375,1.0,6.5,1.5,1.7,2.0
1,1998-01-01,1.5,3.2,7.2500,1.6353,1.4,6.4,1.7,1.9,2.0,3.4,1.6597,5.9375,1.4,6.4,1.6,1.8,1.7
2,1998-02-01,1.6,2.9,7.2500,1.6407,1.4,6.4,1.5,1.9,2.0,3.2,1.6353,7.2500,1.4,6.4,1.7,2.0,1.4
3,1998-03-01,1.7,3.3,7.2500,1.6620,0.8,6.3,1.6,1.7,1.8,2.9,1.6407,7.2500,1.4,6.4,1.8,1.7,1.3
4,1998-04-01,1.8,10.0,7.2500,1.6733,0.9,6.3,1.7,1.5,1.9,3.3,1.6620,7.2500,0.8,6.3,2.0,1.4,1.4


In [11]:
from sklearn.preprocessing import MinMaxScaler

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Fit on training part only
scaler_X.fit(train_df[feature_cols])
scaler_y.fit(train_df[target_cols])

# Transform full series (for sequence building later)
X_all_scaled = scaler_X.transform(df[feature_cols])
y_all_scaled = scaler_y.transform(df[target_cols])


In [12]:
lookback = 12

def create_sequences(X, y, lookback):
    X_seq, y_seq, idx = [], [], []
    for i in range(lookback, len(X)):
        X_seq.append(X[i - lookback:i, :])
        y_seq.append(y[i, :])
        idx.append(i)
    return np.array(X_seq), np.array(y_seq), np.array(idx)

X_seq, y_seq, idx_positions = create_sequences(X_all_scaled, y_all_scaled, lookback)

dates = df['date'].values
seq_dates = dates[idx_positions]

train_mask_seq = seq_dates <= np.datetime64(train_end)
val_mask_seq   = (seq_dates > np.datetime64(train_end)) & (seq_dates <= np.datetime64(val_end))
test_mask_seq  = seq_dates > np.datetime64(val_end)

X_train_seq = X_seq[train_mask_seq]
y_train_seq = y_seq[train_mask_seq]

X_val_seq = X_seq[val_mask_seq]
y_val_seq = y_seq[val_mask_seq]

X_test_seq = X_seq[test_mask_seq]
y_test_seq = y_seq[test_mask_seq]

print("Train seq:", X_train_seq.shape, "Val seq:", X_val_seq.shape, "Test seq:", X_test_seq.shape)


Train seq: (229, 12, 14) Val seq: (48, 12, 14) Test seq: (38, 12, 14)


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

n_features = X_train_seq.shape[2]
n_targets  = y_train_seq.shape[1]

lstm_model = Sequential()
lstm_model.add(LSTM(64, input_shape=(lookback, n_features)))
lstm_model.add(Dense(32, activation='relu'))
lstm_model.add(Dense(n_targets))   # 3 outputs: cpi_t1, cpi_t3, cpi_t5

lstm_model.compile(optimizer='adam', loss='mse')

es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = lstm_model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=100,
    batch_size=16,
    callbacks=[es],
    verbose=1
)


Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - loss: 0.1065 - val_loss: 0.1110
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0286 - val_loss: 0.1053
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0214 - val_loss: 0.1002
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0209 - val_loss: 0.1018
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0184 - val_loss: 0.0898
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0156 - val_loss: 0.0893
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0158 - val_loss: 0.0925
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0143 - val_loss: 0.0883
Epoch 9/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0140 - val_loss: 0.0887
Epoch 10/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0152 - val_loss: 0.0852
Epoch 11/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0127 - val_loss: 0.0804
Epoch 12/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0

In [14]:
# Predict for all sequences
lstm_pred_scaled = lstm_model.predict(X_seq)
lstm_pred = scaler_y.inverse_transform(lstm_pred_scaled)

# Copy original df and add prediction columns
df_hybrid = df.copy()
for i, col in enumerate(target_cols):
    df_hybrid.loc[idx_positions, f'lstm_{col}'] = lstm_pred[:, i]

# Remove rows without LSTM predictions (first 'lookback' months)
df_hybrid = df_hybrid.dropna(subset=[f'lstm_{c}' for c in target_cols]).reset_index(drop=True)

df_hybrid[['date','cpi_t1','cpi_t3','cpi_t5',
           'lstm_cpi_t1','lstm_cpi_t3','lstm_cpi_t5']].head()


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step


,date,cpi_t1,cpi_t3,cpi_t5,lstm_cpi_t1,lstm_cpi_t3,lstm_cpi_t5
0,1998-12-01,1.6,1.7,1.3,1.276856,1.451091,1.025874
1,1999-01-01,1.4,1.5,1.3,1.251589,1.460514,1.015553
2,1999-02-01,1.7,1.3,1.3,1.251375,1.457354,1.043950
3,1999-03-01,1.5,1.3,1.2,1.218872,1.452710,1.079867
4,1999-04-01,1.3,1.3,1.2,1.213359,1.473941,1.112436


In [15]:
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

# Rebuild split masks on df_hybrid
train_mask_h = df_hybrid['date'] <= train_end
val_mask_h   = (df_hybrid['date'] > train_end) & (df_hybrid['date'] <= val_end)
test_mask_h  = df_hybrid['date'] > val_end

hybrid_feature_cols = feature_cols + [f'lstm_{c}' for c in target_cols]

X_train_h = df_hybrid.loc[train_mask_h, hybrid_feature_cols]
y_train_h = df_hybrid.loc[train_mask_h, target_cols]

X_val_h   = df_hybrid.loc[val_mask_h, hybrid_feature_cols]
y_val_h   = df_hybrid.loc[val_mask_h, target_cols]

X_test_h  = df_hybrid.loc[test_mask_h, hybrid_feature_cols]
y_test_h  = df_hybrid.loc[test_mask_h, target_cols]

base_xgb = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

hybrid_model = MultiOutputRegressor(base_xgb)
hybrid_model.fit(X_train_h, y_train_h)

hybrid_pred = hybrid_model.predict(X_test_h)

hybrid_mae = mean_absolute_error(y_test_h, hybrid_pred)
hybrid_rmse = np.sqrt(mean_squared_error(y_test_h, hybrid_pred))

print("Hybrid LSTM + XGBoost MAE :", hybrid_mae)
print("Hybrid LSTM + XGBoost RMSE:", hybrid_rmse)


Hybrid LSTM + XGBoost MAE : 3.176072120666504
Hybrid LSTM + XGBoost RMSE: 4.142076000553388
